In [1]:
from langchain_classic.retrievers import ParentDocumentRetriever
from langchain_classic.storage import InMemoryStore
from langchain_chroma import Chroma
from langchain_ollama import OllamaEmbeddings
from langchain_community.document_loaders import AsyncHtmlLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

/var/folders/qp/9vxvmncx0ks8cprx94py8fdh0000gn/T/ipykernel_50055/3598089307.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import AsyncHtmlLoader
USER_AGENT environment variable not set, consider setting it to identify your requests.


In [2]:
parent_splitter = RecursiveCharacterTextSplitter(chunk_size=3000)
child_splitter = RecursiveCharacterTextSplitter(chunk_size=500)

EMBEDDING_MODEL = 'nomic-embed-text:latest'
child_chunks_collection = Chroma(
    collection_name="uk_child_chunks",
    embedding_function=OllamaEmbeddings(model=EMBEDDING_MODEL),
)
child_chunks_collection.reset_collection() #D
doc_store = InMemoryStore() #E

parent_doc_retriever = ParentDocumentRetriever( #F
    vectorstore=child_chunks_collection,
    docstore=doc_store,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter,
    search_kwargs={"k": 8}, # Changes the number of child chunks retrieved to 8
)

In [3]:
uk_destinations = [
    "Cornwall", "North_Cornwall", "South_Cornwall", "West_Cornwall", 
    "Tintagel", "Bodmin", "Wadebridge", "Penzance", "Newquay",
    "St_Ives", "Port_Isaac", "Looe", "Polperro", "Porthleven"
    "East_Sussex", "Brighton", "Battle", "Hastings_(England)", 
    "Rye_(England)", "Seaford", "Ashdown_Forest"
] 

wikivoyage_root_url = "https://en.wikivoyage.org/wiki"
uk_destination_urls = [f'{wikivoyage_root_url}/{d}' for d in uk_destinations]
from langchain_community.document_transformers import Html2TextTransformer
html2text_transformer = Html2TextTransformer()

for destination_url in uk_destination_urls:
    html_loader = AsyncHtmlLoader(destination_url) #A
    html_docs =  html_loader.load() #B
    text_docs = html2text_transformer.transform_documents(html_docs) #C

    print(f'Ingesting {destination_url}')
    parent_doc_retriever.add_documents(text_docs, ids=None) #D


Fetching pages: 100%|##################################################| 1/1 [00:00<00:00,  3.79it/s]


Ingesting https://en.wikivoyage.org/wiki/Cornwall


Fetching pages: 100%|##################################################| 1/1 [00:00<00:00,  9.45it/s]


Ingesting https://en.wikivoyage.org/wiki/North_Cornwall


Fetching pages: 100%|##################################################| 1/1 [00:00<00:00,  9.59it/s]


...

Ingesting https://en.wikivoyage.org/wiki/Rye_(England)


Fetching pages: 100%|##################################################| 1/1 [00:00<00:00,  8.51it/s]


Ingesting https://en.wikivoyage.org/wiki/Seaford


Fetching pages: 100%|##################################################| 1/1 [00:00<00:00,  5.75it/s]


Ingesting https://en.wikivoyage.org/wiki/Ashdown_Forest


In [4]:
retrieved_docs = parent_doc_retriever.invoke("Cornwall Ranger")
print(retrieved_docs[0])

page_content='**Go Cornwall Bus** buses operate between:

  * **10** \- Plymouth to Saltash, Looe and Polperro
  * **11** \- Plymouth to Saltash, Liskeard, Bodmin, Wadebridge and Padstow

**Stagecoach** buses operate between Barnstaple, Holsworthy, Launceston and
Tavistock, across the Cornwall and Devon border (**85**).

## Get around

[edit]

### By bus

[edit]

Thanks to Transport for Cornwall, all bus tickets are interchangeable across
the different companies (except certain town buses in St Ives and Fowey). The
**Cornwall All Day ticket** allows unlimited travel for a calendar day. As of
2025, day passes are £8 for adults and £5 for under-19s and £3 singles,
regardless of age. Payment is by cash or contactless. Real time information
and timetables can now be found through Transport for Cornwall (most reliable
for real time info), Transit (flaky & unreliable at times, and only buses) or
Citymapper (shows all public transport in Cornwall, yet sometimes unreliable
for real time info).

In [5]:
child_docs_only =  child_chunks_collection.similarity_search("Cornwall Ranger")
print(child_docs_only[0])

page_content='The **Cornwall Ranger** ticket allows unlimited train travel in Cornwall and
Plymouth for a calendar day. As of 2023, this costs £14 for adults and £7 for
under-16s.

There is also a new Pay as you Go smartcard system for Cornwall, run by GWR
but valid on all trains in Cornwall.

### By ferry/boat

[edit]' metadata={'language': 'en', 'source': 'https://en.wikivoyage.org/wiki/Cornwall', 'title': 'Cornwall – Travel guide at Wikivoyage', 'doc_id': 'b32ea84a-683d-4a32-8a53-bb79b778fcb5'}
